### Import libraries

In [5]:
from pathlib import Path
import json, pandas as pd, joblib
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

### Load datasets

In [6]:
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "datasets/processed"

train = pd.read_csv(
    DATA / "paddy_train.csv",
    keep_default_na=False,
    encoding="utf-8-sig"
)

val = pd.read_csv(
    DATA / "paddy_validation.csv",
    keep_default_na=False,
    encoding="utf-8-sig"
)

### Extract text features

In [7]:
features = FeatureUnion([
    ("char", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=250000,
        sublinear_tf=True
    )),
    ("word", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=100000,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w+\b"
    )),
])

### machine learning model

In [8]:
model = Pipeline([
    ("features", features),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(train["question"], train["label"])
pred = model.predict(val["question"])

### evaluation metrics

In [9]:
precision, recall, f1_macro, _ = precision_recall_fscore_support(
    val["label"],
    pred,
    average="macro",
    zero_division=0
)

metrics = {
    "model": "tfidf_linear_svc",
    "accuracy": float(accuracy_score(val["label"], pred)),
    "precision_macro": float(precision),
    "recall_macro": float(recall),
    "f1_macro": float(f1_macro)
}

print(json.dumps(metrics, indent=2))

print(classification_report(val["label"], pred, zero_division=0))

{
  "model": "tfidf_linear_svc",
  "accuracy": 0.4549479166666667,
  "precision_macro": 0.3507709599641389,
  "recall_macro": 0.37743055555555555,
  "f1_macro": 0.34154317448137733
}
                         precision    recall  f1-score   support

     Disease management       0.41      0.75      0.53       480
Farm operations & costs       0.00      0.00      0.00       240
 Harvest & post-harvest       0.75      0.75      0.75       480
            Land & soil       0.00      0.00      0.00       240
     Nursery & planting       0.00      0.00      0.00       240
 Nutrients & fertilizer       0.38      0.50      0.43       240
        Pest management       0.32      0.50      0.39       480
   Safety & environment       0.00      0.00      0.00       240
         Seed & variety       0.00      0.00      0.00       240
       Water management       0.75      0.75      0.75       480
      Weather & climate       0.94      0.30      0.45       240
        Weed management       0.67  

### Save model

In [10]:
model_dir = ROOT / "models/problem_classifier"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_dir / "tfidf_linear_svc.joblib")

(ROOT / "models/model_evaluation_results/baseline_validation_metrics.json").write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

pd.DataFrame({
    "question": val["question"],
    "language": val["language"],
    "issue_id": val["issue_id"],
    "true_label": val["label"],
    "predicted_label": pred
}).to_csv(
    ROOT / "models/model_evaluation_results/baseline_validation_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

